# Exercise 1 — Custom JSON Serialization

Build a robust `json.JSONEncoder` that can serialize dictionaries containing `Stock` and `Trade` objects while preserving enough type information for a future decoder.

### Design goals
- Preserve exact `Decimal` values (serialize them as strings, not floats).
- Preserve `date` and `datetime` types using ISO-8601 text plus explicit type markers.
- Add an explicit object discriminator (`__type__`) so `Stock` and `Trade` can be reconstructed later.
- Add a schema version (`__version__`) for forward-compatible evolution.
- Fail loudly for unsupported objects instead of silently producing lossy data.


## 1. Domain classes and sample data

In [1]:
from datetime import date, datetime
from decimal import Decimal
import json
from typing import Any, Optional


class Stock:
    def __init__(self, symbol, date, open_, high, low, close, volume):
        self.symbol = symbol
        self.date = date
        self.open = open_
        self.high = high
        self.low = low
        self.close = close
        self.volume = volume

    def __repr__(self):
        return (
            f"Stock(symbol={self.symbol!r}, date={self.date!r}, "
            f"open={self.open!r}, high={self.high!r}, low={self.low!r}, "
            f"close={self.close!r}, volume={self.volume!r})"
        )


class Trade:
    def __init__(self, symbol, timestamp, order, price, volume, commission):
        self.symbol = symbol
        self.timestamp = timestamp
        self.order = order
        self.price = price
        self.commission = commission
        self.volume = volume

    def __repr__(self):
        return (
            f"Trade(symbol={self.symbol!r}, timestamp={self.timestamp!r}, "
            f"order={self.order!r}, price={self.price!r}, "
            f"volume={self.volume!r}, commission={self.commission!r})"
        )


In [2]:
activity = {
    "quotes": [
        Stock(
            "TSLA",
            date(2018, 11, 22),
            Decimal("338.19"),
            Decimal("338.64"),
            Decimal("337.60"),
            Decimal("338.19"),
            365_607,
        ),
        Stock(
            "AAPL",
            date(2018, 11, 22),
            Decimal("176.66"),
            Decimal("177.25"),
            Decimal("176.64"),
            Decimal("176.78"),
            3_699_184,
        ),
        Stock(
            "MSFT",
            date(2018, 11, 22),
            Decimal("103.25"),
            Decimal("103.48"),
            Decimal("103.07"),
            Decimal("103.11"),
            4_493_689,
        ),
    ],
    "trades": [
        Trade(
            "TSLA",
            datetime(2018, 11, 22, 10, 5, 12),
            "buy",
            Decimal("338.25"),
            100,
            Decimal("9.99"),
        ),
        Trade(
            "AAPL",
            datetime(2018, 11, 22, 10, 30, 5),
            "sell",
            Decimal("177.01"),
            20,
            Decimal("9.99"),
        ),
    ],
}

activity

{'quotes': [Stock(symbol='TSLA', date=datetime.date(2018, 11, 22), open=Decimal('338.19'), high=Decimal('338.64'), low=Decimal('337.60'), close=Decimal('338.19'), volume=365607),
  Stock(symbol='AAPL', date=datetime.date(2018, 11, 22), open=Decimal('176.66'), high=Decimal('177.25'), low=Decimal('176.64'), close=Decimal('176.78'), volume=3699184),
  Stock(symbol='MSFT', date=datetime.date(2018, 11, 22), open=Decimal('103.25'), high=Decimal('103.48'), low=Decimal('103.07'), close=Decimal('103.11'), volume=4493689)],
 'trades': [Trade(symbol='TSLA', timestamp=datetime.datetime(2018, 11, 22, 10, 5, 12), order='buy', price=Decimal('338.25'), volume=100, commission=Decimal('9.99')),
  Trade(symbol='AAPL', timestamp=datetime.datetime(2018, 11, 22, 10, 30, 5), order='sell', price=Decimal('177.01'), volume=20, commission=Decimal('9.99'))]}

## 2. Serialization format

A plain `obj.__dict__` dump is intentionally avoided because it loses type information and couples the JSON representation to implementation details.

Instead, every non-native JSON type is encoded with an explicit `__type__` discriminator. Domain objects also carry `__version__`, which gives a future decoder a safe place to handle schema migrations.

Example shape:

```json
{
  "__type__": "Decimal",
  "value": "338.19"
}
```


In [3]:
TYPE_FIELD = "__type__"
VERSION_FIELD = "__version__"
SCHEMA_VERSION = 1


class FinancialJSONEncoder(json.JSONEncoder):
    """JSON encoder for Stock/Trade domain objects and their value types.

    The representation is deliberately explicit and lossless:
    - Decimal -> tagged string value (avoids float precision loss)
    - datetime -> tagged ISO-8601 value
    - date -> tagged ISO-8601 value
    - Stock/Trade -> tagged field dictionaries with schema version

    Unknown objects are delegated to json.JSONEncoder.default(), which
    raises TypeError in the normal way.
    """

    def default(self, obj: Any) -> Any:
        # datetime must be checked before date because datetime is a
        # subclass of date.
        if isinstance(obj, datetime):
            return {
                TYPE_FIELD: "datetime",
                "value": obj.isoformat(),
            }

        if isinstance(obj, date):
            return {
                TYPE_FIELD: "date",
                "value": obj.isoformat(),
            }

        if isinstance(obj, Decimal):
            return {
                TYPE_FIELD: "Decimal",
                "value": str(obj),
            }

        if isinstance(obj, Stock):
            return {
                TYPE_FIELD: "Stock",
                VERSION_FIELD: SCHEMA_VERSION,
                "symbol": obj.symbol,
                "date": obj.date,
                "open": obj.open,
                "high": obj.high,
                "low": obj.low,
                "close": obj.close,
                "volume": obj.volume,
            }

        if isinstance(obj, Trade):
            return {
                TYPE_FIELD: "Trade",
                VERSION_FIELD: SCHEMA_VERSION,
                "symbol": obj.symbol,
                "timestamp": obj.timestamp,
                "order": obj.order,
                "price": obj.price,
                "volume": obj.volume,
                "commission": obj.commission,
            }

        return super().default(obj)


## 3. Small public helper

Keeping encoder configuration in one helper prevents callers from repeatedly specifying options and makes the serialization contract easier to change later.

In [4]:
def serialize_activity(data: Any, indent: Optional[int] = 2) -> str:
    """Serialize supported financial-domain data to JSON text."""
    return json.dumps(
        data,
        cls=FinancialJSONEncoder,
        indent=indent,
        ensure_ascii=False,
        sort_keys=True,
    )


## 4. Serialize the provided object

In [5]:
serialized_activity = serialize_activity(activity)
print(serialized_activity)

{
  "quotes": [
    {
      "__type__": "Stock",
      "__version__": 1,
      "close": {
        "__type__": "Decimal",
        "value": "338.19"
      },
      "date": {
        "__type__": "date",
        "value": "2018-11-22"
      },
      "high": {
        "__type__": "Decimal",
        "value": "338.64"
      },
      "low": {
        "__type__": "Decimal",
        "value": "337.60"
      },
      "open": {
        "__type__": "Decimal",
        "value": "338.19"
      },
      "symbol": "TSLA",
      "volume": 365607
    },
    {
      "__type__": "Stock",
      "__version__": 1,
      "close": {
        "__type__": "Decimal",
        "value": "176.78"
      },
      "date": {
        "__type__": "date",
        "value": "2018-11-22"
      },
      "high": {
        "__type__": "Decimal",
        "value": "177.25"
      },
      "low": {
        "__type__": "Decimal",
        "value": "176.64"
      },
      "open": {
        "__type__": "Decimal",
        "value": "176.66"
   

## 5. Validation checks

These checks do **not** deserialize into `Stock` or `Trade` yet—that belongs to Exercise 2. They only verify that Exercise 1 produces valid JSON and includes the metadata needed for lossless reconstruction.

In [6]:
raw = json.loads(serialized_activity)

# The JSON itself is valid and has the expected top-level structure.
assert set(raw) == {"quotes", "trades"}
assert len(raw["quotes"]) == 3
assert len(raw["trades"]) == 2

# Domain type and schema metadata are present.
assert raw["quotes"][0][TYPE_FIELD] == "Stock"
assert raw["quotes"][0][VERSION_FIELD] == SCHEMA_VERSION
assert raw["trades"][0][TYPE_FIELD] == "Trade"
assert raw["trades"][0][VERSION_FIELD] == SCHEMA_VERSION

# Decimal precision is preserved as text rather than converted to float.
assert raw["quotes"][0]["open"] == {
    TYPE_FIELD: "Decimal",
    "value": "338.19",
}

# date and datetime remain distinguishable.
assert raw["quotes"][0]["date"] == {
    TYPE_FIELD: "date",
    "value": "2018-11-22",
}
assert raw["trades"][0]["timestamp"] == {
    TYPE_FIELD: "datetime",
    "value": "2018-11-22T10:05:12",
}

print("All serialization checks passed.")

All serialization checks passed.


## 6. Unsupported-type behavior

A good custom encoder should not silently stringify arbitrary objects. Delegating unknown values to the base encoder preserves the standard `TypeError` behavior.

In [7]:
class Unsupported:
    pass


try:
    json.dumps(Unsupported(), cls=FinancialJSONEncoder)
except TypeError as exc:
    print(f"Expected failure for unsupported type: {exc}")

Expected failure for unsupported type: Object of type Unsupported is not JSON serializable


## Result

`FinancialJSONEncoder` now provides a deterministic, explicit, and lossless JSON representation for the supplied `Stock`/`Trade` structure. The `__type__` markers and version field are intentionally designed so Exercise 2 can implement the reverse transformation cleanly.